# K-Nearest-Neighbors

Here is the implementation of the second collaborative model for recommending videos. The aim is to provide a new model to address cold-start issues for users. Indeed, when we have a new user that has not seen any video yet, we need to recommend videos with a little amount of data under our hands.

To address this issue, we are going to set a context setting: given that we are coding a recommender system for an app such as Tiktok or YouTube, even though the user has not seen any video, we already have data about the user from 2 sources:

- the platform requires or push the user to create an account with general data about himself/herself such as the age, the address, the country or about the gender. For instance, Netflix requires to have an account and Tiktok forces and push the user to create one.
- the application works with external partners to collect data about the user and get a "numerical footprint" that can be used to recommend videos. For instance, YouTube uses data collected from everywhere because it benefits from Google Ads and Google tracking features that knows the taste of the users. 

Given this setting, we can assume we have the minimal require amount of data to compute similarities between users to recommend videos. The KuaiRec dataset provides us encrypted features about users. We do not know what does they represent but we can make the assumptions that users can form groups by computing their similarities.

This leads us to use K-Nearest-Neighbors where we will compute similarities of our new user against all the existing ones in our database. Then we will recommend the most liked videos by the most similar users.

## Import libraries

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import sys
sys.path.append('../')
import metrics

MANUAL_RANDOM_SEED=42
EMBEDDINGS_PATH='../embeddings'

## Load embeddings

In [2]:
user_ratings: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_ratings.pkl')
user_encrypted: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_encrypted.pkl')

In [3]:
# Set user_id as index to manipulate the dataframe better
user_ratings = user_ratings.set_index('user_id')

## Preprocess data

We need some preprocessing before using a custom KNN model. Let's use the first user to find out how to compute similarities in order to build the model.

In [4]:
# Get user encrypted features (this can be geographical location for instance, we do not know!)
user = user_encrypted.iloc[0]
user

onehot_feat0                    0
onehot_feat1                    1
onehot_feat2                   17
onehot_feat3                  638
onehot_feat4                  2.0
onehot_feat5                    0
onehot_feat6                    1
onehot_feat7                    6
onehot_feat8                  184
onehot_feat9                    6
onehot_feat10                   3
onehot_feat11                   0
onehot_feat12                 0.0
onehot_feat13                 0.0
onehot_feat14                 0.0
onehot_feat15                 0.0
onehot_feat16                 0.0
onehot_feat17                 0.0
user_active_degree    high_active
Name: 0, dtype: object

In [5]:
# Let's drop user_active_degree because we not need it to compute
# similarities between users, however we will need this feature later
user.drop('user_active_degree', axis=0).to_numpy().reshape(-1, 1)

array([[0],
       [1],
       [17],
       [638],
       [2.0],
       [0],
       [1],
       [6],
       [184],
       [6],
       [3],
       [0],
       [0.0],
       [0.0],
       [0.0],
       [0.0],
       [0.0],
       [0.0]], dtype=object)

In [6]:
# Compute similarities between our user and all others
user_similarities = cosine_similarity([user.drop('user_active_degree', axis=0).to_numpy()], user_encrypted.drop('user_active_degree', axis=1))
# We assume that we will get a 1 at then end of the array, because
# we compute the similarity of the user with itself
user_similarities.sort()
user_similarities

array([[0.27824314, 0.2807758 , 0.2811921 , ..., 0.99999238, 0.99999837,
        1.        ]])

## Implementing the KNN model

Here are the steps for implementing the model by hand:

1. Remove the user from the one I will compute the similarities on
2. Keep only the highest active users that is the one with the field `user_active_degree` is set on `high_active`. I am doing that because I want to recommend to new users videos that has been seen recently. It enables the predictions to follow the last trends on the platform without taking account of old users that might watched very old videos that do not match today tastes.
3. Compute similarities between the given user with all the others
4. Get the top users and get the video they liked
5. Recommend the videos that the most users liked that given 20 retrieved users, I will highly recommend a video that all the similar users liked.

For evaluating the model we just need a tiny adjustment: we want to recommend videos that the given user has already seen just for the purpose of having a comparison value to compute the metrics on.

In [7]:
# Implementation of the KNN recommender algorithm
def knn_recommend_videos(user_id: int, user_data: pd.DataFrame, users: pd.DataFrame, k: int = 5, validation: bool = False):
    users = users[users.index != user_id]
    # Keep highly active users
    users = users[users['user_active_degree'] == 'high_active']
    users = users.drop('user_active_degree', axis=1)

    # Compute similarities
    similarities = cosine_similarity([user_data], users).ravel()
    # Get top 20 users
    similarities = similarities.argsort()[::-1][:20]
    similar_users = users.iloc[similarities]

    # Retrieve their ratings
    similar_user_ratings = user_ratings.loc[similar_users.index]
    similar_user_ratings = similar_user_ratings.reset_index()

    if validation:
        # Filter by videos seen by the given user to evaluate the model
        similar_user_ratings = similar_user_ratings[similar_user_ratings['video_id'].isin(user_ratings.loc[user_id]['video_id'])]

    # Keep only the video that users liked
    similar_user_ratings = similar_user_ratings[similar_user_ratings['like'] == 1]

    # Get most common liked videos
    return similar_user_ratings['video_id'].value_counts().index[:k].to_list()


In [8]:
# Recommend video for our first user example above
knn_recommend_videos(0, user.drop('user_active_degree', axis=0).to_numpy(), user_encrypted, validation=True)

[10435, 9195, 4411, 4506, 7383]

## Evaluation

As this is an unsupervised model, we do not have any training phase so we can move on evaluating the actual knn model. I build the recommendations matrix that stores the video recommendations for each user and then apply the metrics on to measure the performance of the model.

In [9]:
recommendations = None

for user_id, user_data in user_encrypted.groupby('user_id'):
    print(f'Current user is: {user_id}')
    user_data = user_data.drop('user_active_degree', axis=1).to_numpy().ravel()
    recommendation = knn_recommend_videos(user_id, user_data, user_encrypted, k=20, validation=True)
    video_recommended = user_ratings.loc[user_id]
    video_recommended = video_recommended[video_recommended['video_id'].isin(recommendation)]

    if recommendations is None:
        recommendations = video_recommended
    else:
        recommendations = pd.concat([recommendations, video_recommended])

Current user is: 0
Current user is: 1
Current user is: 2
Current user is: 3
Current user is: 4
Current user is: 5
Current user is: 6
Current user is: 7
Current user is: 8
Current user is: 9
Current user is: 10
Current user is: 11
Current user is: 12
Current user is: 13
Current user is: 14
Current user is: 15
Current user is: 17
Current user is: 18
Current user is: 19
Current user is: 20
Current user is: 21
Current user is: 22
Current user is: 23
Current user is: 24
Current user is: 25
Current user is: 26
Current user is: 27
Current user is: 28
Current user is: 29
Current user is: 30
Current user is: 31
Current user is: 32
Current user is: 33
Current user is: 34
Current user is: 35
Current user is: 36
Current user is: 37
Current user is: 38
Current user is: 39
Current user is: 40
Current user is: 43
Current user is: 44
Current user is: 45
Current user is: 46
Current user is: 47
Current user is: 48
Current user is: 49
Current user is: 50
Current user is: 51
Current user is: 52
Current us

In [10]:
recommendations.head()

,video_id,like,watch_ratio
user_id,,,
0,5853,1,2.587314
0,8435,1,3.525128
0,10021,1,1.477522
0,7383,1,4.318226
0,2629,0,0.144295


In [ ]:
# Select columns in the recommendation dataframe to compute the diversity metric on
video_content: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_content.pkl')
video_tags: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_one_hot_tags.pkl')

recommendations = recommendations.reset_index()
recommendations = recommendations.merge(video_tags, on='video_id')
recommendations = recommendations.merge(video_content.iloc[:, 2:], on='video_id')

recommendations.head() 

,user_id,video_id,like,watch_ratio,tag_0,tag_1,tag_2,tag_3,tag_4,tag_5,...,tag_28,tag_29,tag_30,show_cnt,play_cnt,valid_play_cnt,like_cnt,comment_cnt,share_cnt,video_duration
0,0,5853,1,2.587314,0,0,0,0,0,0,...,1,0,0,809435,817112,460540,10079,95,11,5234
1,0,8435,1,3.525128,0,0,0,0,0,0,...,0,0,0,81061,84472,68445,466,0,0,3900
2,0,10021,1,1.477522,0,0,0,0,0,0,...,0,0,0,3095234,948298,388491,49343,3130,279,8475
3,0,7383,1,4.318226,0,0,0,0,0,0,...,0,0,0,384622,393672,310240,2078,12,3,3067
4,0,2629,0,0.144295,0,0,0,0,0,0,...,0,0,0,958437,887335,590171,75657,1582,39,5267


In [ ]:
# Select columns in the recommendation dataframe to compute the diversity metric on
video_content_columns = recommendations.columns[4:]
video_content_columns

Index(['tag_0', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7',
       'tag_8', 'tag_9', 'tag_10', 'tag_11', 'tag_12', 'tag_13', 'tag_14',
       'tag_15', 'tag_16', 'tag_17', 'tag_18', 'tag_19', 'tag_20', 'tag_21',
       'tag_22', 'tag_23', 'tag_24', 'tag_25', 'tag_26', 'tag_27', 'tag_28',
       'tag_29', 'tag_30', 'show_cnt', 'play_cnt', 'valid_play_cnt',
       'like_cnt', 'comment_cnt', 'share_cnt', 'video_duration'],
      dtype='object')

In [14]:
# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)
    diversity = metrics.inter_list_diversity_at_k(recommendations, video_content_columns, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print(f'ILD@{k} -> {diversity}')
    print()

Precision@5 -> 0.7058414262936658
NDCG@5 -> 0.8588479725319983
ILD@5 -> 0.9369053860677685

Precision@10 -> 0.6969415857370633
NDCG@10 -> 0.8647343765683367
ILD@10 -> 0.9395781555695202

Precision@15 -> 0.6926511088563561
NDCG@15 -> 0.86678049127918
ILD@15 -> 0.9395422756394614

Precision@20 -> 0.6931946658936078
NDCG@20 -> 0.8689730607383425
ILD@20 -> 0.9403994849321557



## Interpretations

As we can see, the model top up with Precision@5 at 70%. This is actual a very good result given that we have very limited knowledge about the user. This model will be actually very useful if the user is new that is for example if he/she did not like at least 5 or 10 videos. Furthermore, we can see that KNN is the model with the highest diversity metric so far compare to the other models I implemented showing that users tend to like videos that are not popular at first or they do not seem to like a certain category of video.

I will use the model in a certain context (so if the user is new to the recommender system) and then switch to stronger model when we collect enough data about the video taste of the user.